In [1]:
def estimate_kv_cache_bytes(
    num_layers,
    batch_size,
    seq_len,
    num_kv_heads,
    head_dim,
    dtype_bytes=2,  # fp16/bf16
):
    # 2 because key + value
    return 2 * num_layers * batch_size * seq_len * num_kv_heads * head_dim * dtype_bytes

In [2]:
def bytes_to_gb(x):
    return x / 1024**3

In [3]:
from transformers import AutoConfig

model_name = "Qwen/Qwen3-0.6B"
config = AutoConfig.from_pretrained(model_name)

num_layers = config.num_hidden_layers
num_attention_heads = config.num_attention_heads
num_kv_heads = config.num_key_value_heads
head_dim = config.hidden_size // config.num_attention_heads

print("num_layers:", num_layers)
print("num_attention_heads:", num_attention_heads)
print("num_kv_heads:", num_kv_heads)
print("head_dim:", head_dim)

/home/mv/miniconda3/envs/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_layers: 28
num_attention_heads: 16
num_kv_heads: 8
head_dim: 64


In [4]:
seq_lens = [512, 2048, 8192, 32768]
batch_sizes = [1, 4, 16, 64]

for batch_size in batch_sizes:
    print(f"\nBatch size = {batch_size}")

    for seq_len in seq_lens:
        kv_bytes = estimate_kv_cache_bytes(
            num_layers=num_layers,
            batch_size=batch_size,
            seq_len=seq_len,
            num_kv_heads=num_kv_heads,
            head_dim=head_dim,
            dtype_bytes=2,
        )

        print(
            f"seq_len={seq_len:>6} | "
            f"KV cache={bytes_to_gb(kv_bytes):.3f} GB"
        )


Batch size = 1
seq_len=   512 | KV cache=0.027 GB
seq_len=  2048 | KV cache=0.109 GB
seq_len=  8192 | KV cache=0.438 GB
seq_len= 32768 | KV cache=1.750 GB

Batch size = 4
seq_len=   512 | KV cache=0.109 GB
seq_len=  2048 | KV cache=0.438 GB
seq_len=  8192 | KV cache=1.750 GB
seq_len= 32768 | KV cache=7.000 GB

Batch size = 16
seq_len=   512 | KV cache=0.438 GB
seq_len=  2048 | KV cache=1.750 GB
seq_len=  8192 | KV cache=7.000 GB
seq_len= 32768 | KV cache=28.000 GB

Batch size = 64
seq_len=   512 | KV cache=1.750 GB
seq_len=  2048 | KV cache=7.000 GB
seq_len=  8192 | KV cache=28.000 GB
seq_len= 32768 | KV cache=112.000 GB


In [5]:
def print_kv_memory_style(name, num_kv_heads):
    print(f"\n{name} | num_kv_heads = {num_kv_heads}")

    for seq_len in [2048, 8192, 32768]:
        kv_bytes = estimate_kv_cache_bytes(
            num_layers=num_layers,
            batch_size=1,
            seq_len=seq_len,
            num_kv_heads=num_kv_heads,
            head_dim=head_dim,
            dtype_bytes=2,
        )

        print(f"seq_len={seq_len:>6} | {bytes_to_gb(kv_bytes):.3f} GB")


# MHA: every attention head has its own K/V head
print_kv_memory_style("MHA", num_attention_heads)

# GQA: model's actual grouped-query attention setting
print_kv_memory_style("GQA", config.num_key_value_heads)

# MQA: all query heads share one K/V head
print_kv_memory_style("MQA", 1)


MHA | num_kv_heads = 16
seq_len=  2048 | 0.219 GB
seq_len=  8192 | 0.875 GB
seq_len= 32768 | 3.500 GB

GQA | num_kv_heads = 8
seq_len=  2048 | 0.109 GB
seq_len=  8192 | 0.438 GB
seq_len= 32768 | 1.750 GB

MQA | num_kv_heads = 1
seq_len=  2048 | 0.014 GB
seq_len=  8192 | 0.055 GB
seq_len= 32768 | 0.219 GB
